# Choose an appropriate compute type

## Serverless compute

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.select-and-configure-compute/02-serverless-compute.svg)

Key points to cover:
- Managed entirely by Azure Databricks, runs in Databricks' Azure subscription; no VMs appear in customer's subscription
- Startup time: 2-6 seconds; auto-scales up and down to zero
- Available for: Notebooks, Jobs, Pipelines, and SQL Warehouses
- Versionless runtime: Azure Databricks automatically upgrades — no migration effort
- Requires Unity Catalog
- Limitations: No RDD APIs, R language, JAR libraries in notebooks, or custom Spark configurations

## Classic compute

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.select-and-configure-compute/02-classic-compute.svg)

Key points to cover:
- Runs in the customer's Azure subscription — full visibility of underlying infrastructure
- Two access modes: **Standard** (multi-user with Lakeguard isolation) and **Dedicated** (single user or group)
- Dedicated access mode required for RDD APIs, GPU workloads, R language, custom containers
- Two cluster modes: **Multi-node** (driver + workers) and **Single-node** (driver only, no horizontal scaling)
- Startup time: 3-7 minutes; runtime version must be manually selected and upgraded

## SQL warehouses

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.select-and-configure-compute/02-sql-warehouses.svg)

Key points to cover:
- Compute specifically optimized for SQL queries, analytics, and BI workloads
- **Serverless**: 2-6 second startup, Intelligent Workload Management, Photon + Predictive IO
- **Pro**: Custom networking/federation, ~4 min startup, Photon + Predictive IO but no IWM
- **Classic**: Entry-level SQL, basic Photon only, ~4 min startup

## Instance pools

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.select-and-configure-compute/02-instance-pools.svg)

Key points to cover:
- Maintains idle VMs ready for immediate use — reduces startup from minutes to under 1 minute
- Configure: minimum idle instances, maximum capacity, idle auto-termination
- Can preload a Databricks Runtime version for near-instant cluster starts
- Cost: pay for idle VMs (not DBU) — justified only when clusters are created frequently
- Use spot instances for worker nodes; always on-demand for driver node
- Less relevant with serverless available — serverless starts faster without idle capacity overhead

## Job compute

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.select-and-configure-compute/02-job-compute.svg)

Key points to cover:
- Clusters optimized for automated workflows — auto-terminate after job completes
- Choose **serverless** (faster startup, auto-managed, cheaper) or **classic** (more configuration options)
- Spot instances: up to 90% cheaper; Azure reclaims with 30s notice; Spark retries failed tasks automatically
- Spot for worker nodes only — driver must always use on-demand instances
- Job Compute policy: template with best practice defaults including latest LTS runtime

## Compare compute types

Key decision factors:
- **Workload type**: interactive notebooks, SQL queries, or automated jobs
- **APIs/languages**: RDD, R, GPU workloads require Classic Dedicated access mode
- **Frequency**: serverless best for infrequent workloads; instance pools for frequent classic workloads
- **Networking**: custom VNet or on-premises connectivity requires Classic or Pro SQL warehouse
- **Default recommendation**: start with serverless — minimizes overhead, optimizes costs, fastest startup


# Configure compute performance

## Understand compute resource components

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.select-and-configure-compute/03-compute-resource-components.svg)

Key points to cover:
- **Total executor cores**: maximum parallelism — more cores = more parallel tasks simultaneously
- **Total executor memory**: affects in-memory processing; insufficient memory causes spill to disk (major slowdown)
- **Local storage**: temporary space for shuffle operations and caching; fast NVMe storage reduces shuffle time
- These three factors together determine both performance and cost

## Configure node types and cluster size

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.select-and-configure-compute/03-node-types-cluster-size.svg)

Key points to cover:
- **Memory-optimized (E-series)**: large joins, aggregations, in-memory analytics — high RAM per core ratio
- **Compute-optimized (F-series)**: complex CPU calculations, ETL with simple transformations — high CPU, lower memory
- **Storage-optimized (L-series)**: caching workloads, high I/O needs — fast NVMe local storage
- **GPU-accelerated (NC/ND-series)**: ML/deep learning, 10-100x faster than CPU; requires Runtime ML; Photon must be disabled
- Fewer, larger workers: less network shuffle traffic (analytical workloads)
- More, smaller workers: higher parallelism (simple batch processing)

## Configure autoscaling

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.select-and-configure-compute/03-configure-autoscaling.svg)

Key points to cover:
- Set min and max worker counts; Databricks adds/removes workers based on workload demand
- **Optimized autoscaling**: scales up in 2 quick steps; can scale down even when not fully idle (monitors shuffle file state)
- Evaluation intervals: job compute every 40 seconds; all-purpose every 150 seconds
- Best for variable workloads; fixed workers better for predictable, steady-state workloads
- Works best with instance pools: set min workers <= pool minimum idle instances

## Configure termination settings

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.select-and-configure-compute/03-configure-termination.svg)

Key points to cover:
- Specify inactivity period (minutes); cluster terminates if idle beyond the threshold
- Typical timeout for interactive workloads: 45 minutes
- Job compute auto-terminates after job completes; restarts automatically for next scheduled run
- Spot instances: use for worker nodes (Spark handles preemption); driver must always be on-demand
- Enable **decommissioning** with spot workers — migrates shuffle/cached data before instance is reclaimed

## Use instance pools

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.select-and-configure-compute/02-instance-pools.svg)

Key points to cover:
- Minimum idle instances: keep enough warm for concurrent cluster usage needs
- Maximum capacity: controls costs and ensures fair resource distribution across teams
- Idle instance auto-termination: removes excess idle instances after inactivity period
- Preloaded runtime: near-instant cluster starts as runtime already installed on idle instances
- Best ROI for teams with frequent cluster create/destroy cycles throughout the day

## Balance cost and performance

Key points to cover:
- Start conservative, monitor actual usage, adjust based on observed patterns rather than assumptions
- Use **Spark UI** to identify bottlenecks:
  - Jobs Timeline: find long-running stages
  - Stage details: check **Shuffle Spill (Memory)** and **Shuffle Spill (Disk)** statistics
- Spilling to disk or slow queries: increase memory or core count
- Low utilization: reduce cluster size or enable autoscaling
- Use serverless when possible — automatically scales, often best cost-performance without manual tuning


# Configure compute features

## Enable Photon acceleration

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.select-and-configure-compute/04-photon-acceleration.svg)

Key points to cover:
- Photon is a vectorized query engine using optimized native code — replaces standard Spark execution
- Best for: complex joins, aggregations, large table scans, wide tables, repeated transformations
- Minimal benefit for: simple ETL on small datasets, jobs completing in under 2 seconds
- Enabled by default on Databricks Runtime 9.1 LTS+; toggle in the Performance section at cluster creation
- **Not compatible with GPU-enabled clusters** — must disable Photon when using GPU instances

## Select Databricks Runtime and Spark version

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.select-and-configure-compute/04-databricks-runtime.svg)

Key points to cover:
- Runtime includes: Apache Spark, Databricks optimizations, and preloaded libraries
- Selecting a runtime automatically determines the Spark version
- **All-purpose compute**: latest runtime for newest features and package compatibility
- **Job compute**: use LTS versions for extended stability; test before upgrading production
- Upgrade path: test workloads on new runtime in dev environment first, then apply to production

## Configure machine learning environments

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.select-and-configure-compute/04-ml-environments.svg)

Key points to cover:
- **Databricks Runtime ML**: pre-installed ML frameworks (TensorFlow, PyTorch), GPU drivers, CUDA
- Always select Runtime ML (not standard runtime) for ML workloads
- Start with single-node compute for model experimentation — minimal overhead, sufficient resources
- Use GPU instances for deep learning (image recognition, NLP, neural networks) — 10-100x faster than CPU
- Must **disable Photon** when using GPU instances

# Install libraries for compute

## Understand compute-scoped libraries

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.select-and-configure-compute/05-compute-scoped-libraries.svg)

Key points to cover:
- Install on a cluster: available to all notebooks and jobs on that cluster automatically
- Auto-reinstalled every time cluster starts — consistent environment across restarts
- Requires **CAN MANAGE** permission on the cluster
- Supported types: Python wheels, Java JAR files, R packages
- Sources: PyPI, Maven, CRAN, workspace files, Unity Catalog volumes, cloud object storage
- Limitation: affects every notebook on the cluster; conflicting versions require separate clusters

## Install libraries from package repositories

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.select-and-configure-compute/05-libraries-from-repos.svg)

Key points to cover:
- **PyPI**: specify package name; pin exact version for production reproducibility (e.g. pymssql==2.3.9)
- **Maven**: use coordinates format groupId:artifactId:version (e.g. com.microsoft.sqlserver:mssql-jdbc:13.2.1.jre11); can exclude transitive dependencies
- **CRAN (R)**: always installs latest version; to pin versions, use file-based installation
- Standard access mode: Maven coordinates and JAR paths require allowlist approval before installation

## Install libraries from files

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.select-and-configure-compute/05-libraries-from-files.svg)

Key points to cover:
- **Workspace files**: 500 MB limit; upload via Import dialog; path: /Workspace/Users/.../mylib.whl
- **Unity Catalog volumes**: enhanced governance via UC permissions; requires READ VOLUME; path: /Volumes/main/eng/libs/mylib.whl
- **requirements.txt**: install multiple packages from one file (DBR 15.0+); works with workspace files and volumes
- Preferred over ad-hoc pip commands — provides centralized management and audit trails
- Standard access mode: file paths must be in the allowlist

## Use init scripts for advanced configuration

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.select-and-configure-compute/05-init-scripts.svg)

Key points to cover:
- Run shell commands during cluster startup, before Spark driver and executors start
- Use cases: system packages (apt-get), environment variables, monitoring agents, system-level configuration
- **Not recommended** for library installation — use cluster-scoped libraries instead
- Store in Unity Catalog volumes (DBR 13.3+); reference path in cluster configuration
- Non-zero exit code causes cluster startup failure (intentional failure protection)
- Use as a last resort — prefer cluster policies and cluster-scoped libraries

## Configure libraries for standard access mode

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.select-and-configure-compute/05-standard-access-libraries.svg)

Key points to cover:
- Metastore admin must add Maven coordinates and JAR paths to the allowlist before installation on standard access mode clusters
- Allowlist formats: exact version (groupId:artifactId:version), all versions (groupId:artifactId), all artifacts (groupId)
- Path allowlisting uses prefix matching — include trailing slash (e.g. /Volumes/prod-libraries/)
- Init scripts require separate allowlist entries even when stored at the same path as JARs
- Allowlist permission does not grant data access — still need UC permissions (READ VOLUME, etc.)
- Configure in: Catalog Explorer > Metastore settings > Allowed JARs/Init Scripts

# Configure compute access

## Understand compute permission levels

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.select-and-configure-compute/06-compute-permission-levels.svg)

Key points to cover:
- **NO PERMISSIONS**: default state — users cannot see or interact with the compute resource
- **CAN ATTACH TO**: attach notebooks, view Spark UI and metrics — cannot start/stop/restart
  - Avoid No isolation shared access mode: it exposes service account keys to CAN ATTACH TO users (legacy, insecure)
- **CAN RESTART**: all CAN ATTACH TO capabilities + terminate, start, and restart the cluster
- **CAN MANAGE**: full control — edit config, attach libraries, resize, modify others' permissions; workspace admins always have this
- Driver log access: restricted to CAN MANAGE by default (driver logs may expose secrets via stdout/stderr)

## Configure permissions in the workspace

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.select-and-configure-compute/06-configure-permissions.svg)

Key points to cover:
- Configure via workspace UI (not Azure portal): Sidebar > Compute > Permissions
- Add users or groups by name/email; select permission level from dropdown
- Multiple users/groups can hold different permission levels on the same cluster
- Changes take effect immediately — no cluster restart required
- Best practice: **use groups over individuals** — simplifies management; group membership automatically controls access

## Manage workspace-level entitlements

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.select-and-configure-compute/06-workspace-entitlements.svg)

Key points to cover:
- **Unrestricted cluster creation**: allows non-admin users to create clusters without size restrictions
- Does NOT grant workspace admin privileges — follows principle of least privilege
- Grant via: Settings > Identity and access > Users/Groups > Entitlements > Allow cluster creation
- Grant to roles/teams, not individuals; data engineers typically need it, analysts usually do not

## Dedicated group access mode

![diagram](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/demo/mindmaps/learn.wwl.select-and-configure-compute/06-dedicated-group-access.svg)

Key points to cover:
- Assigns compute to a group — user permissions automatically scope down to the group's permissions (prevents privilege escalation)
- Enables secure collaboration on workloads not supported by standard mode: Runtime ML, RDD APIs, R language
- Requirements: Unity Catalog enabled + Databricks Runtime 15.4+; group needs CAN MANAGE on workspace folder
- Setup: create compute with access mode = Dedicated; assign group in 'Single user or group' field
- Objects created on group cluster are owned by the group (not individual users)
- Create group folder at /Workspace/Groups/<groupName> and grant CAN MANAGE to the group
- MLflow tracking URIs and AutoML experiment directories must point to group-accessible folders